# Official CMMD pooled image evaluation

This notebook computes CMMD over **all recursively discovered images** using Google Research's official [`cmmd.main`](https://github.com/google-research/google-research/tree/master/cmmd) script.

Class directories are ignored as labels. Images under `class_0000/`, `class_0001/`, and other nested directories are pooled into one real distribution and one generated distribution.


## 1. CMMD-only dependencies

This notebook needs only the official Google Research and Scenic repositories plus the CMMD requirements:

- `numpy tensorflow flax Pillow absl-py tqdm`
- `torch`, required by Scenic once to convert the downloaded CLIP `.pt` checkpoint to NumPy/JAX variables
- JAX/JAXlib with GPU support for CMMD runs

Set `INSTALL_DEPS = True` once to clone and install the official code. If you already cloned/installed CMMD and only hit the `Could not import torch for CLIP checkpoint conversion` error, set `INSTALL_TORCH = True` once. Keep `REQUIRE_JAX_GPU = True` so preflight fails instead of silently running CMMD on CPU.

For NVIDIA GPUs, the current JAX install guide recommends `pip install -U "jax[cuda13]"`, or `pip install -U "jax[cuda12]"` for CUDA 12 systems. You can let this notebook run that install by setting `INSTALL_JAX_CUDA = True` and choosing `JAX_CUDA_PACKAGE` below. The notebook defaults to clearing `LD_LIBRARY_PATH` for CMMD subprocesses because system CUDA/cuDNN libraries can override JAX's pip-installed libraries and cause cuDNN sublibrary mismatch errors.

### NumPy/Pandas binary mismatch

If you see `ValueError: numpy.dtype size changed`, Pandas and NumPy use incompatible binary ABIs. Set `REPAIR_NUMPY_ABI = True` in the next cell and run it once. The repair preserves the NumPy version selected by TensorFlow/JAX and reinstalls compatible wheels for the compiled packages imported by Keras.


In [ ]:
from __future__ import annotations

import json
import os
import re
import subprocess
import sys
import tempfile
import time
from datetime import datetime
from pathlib import Path

from IPython.display import display

print("Python:", sys.version.split()[0])


In [ ]:
INSTALL_DEPS = False
REPAIR_NUMPY_ABI = False
INSTALL_TORCH = False
TORCH_PACKAGE = "torch"  # CPU torch is enough for Scenic's one-time CLIP checkpoint conversion.
INSTALL_JAX_CUDA = True
JAX_CUDA_PACKAGE = "jax[cuda12]"  # Use "jax[cuda13]" if your NVIDIA driver supports CUDA 13 wheels.
LOCAL_CMMD_PYTHON = Path.cwd() / ".cmmd-env" / "bin" / "python"
CMMD_PYTHON = LOCAL_CMMD_PYTHON if LOCAL_CMMD_PYTHON.is_file() else Path(sys.executable)
TOOLS_DIR = Path.home() / ".cache" / "duodit-metrics"
GOOGLE_RESEARCH_DIR = TOOLS_DIR / "google-research"
SCENIC_DIR = TOOLS_DIR / "scenic"


def checked_run(command: list[str], cwd: Path | None = None) -> None:
    print("Running:", " ".join(command))
    subprocess.run(command, cwd=cwd, check=True)


if INSTALL_DEPS:
    TOOLS_DIR.mkdir(parents=True, exist_ok=True)
    if not GOOGLE_RESEARCH_DIR.exists():
        checked_run([
            "git", "clone", "--depth", "1",
            "https://github.com/google-research/google-research.git",
            str(GOOGLE_RESEARCH_DIR),
        ])
    if not SCENIC_DIR.exists():
        checked_run([
            "git", "clone", "--depth", "1",
            "https://github.com/google-research/scenic.git",
            str(SCENIC_DIR),
        ])
    checked_run([
        str(CMMD_PYTHON), "-m", "pip", "install", "--upgrade", "-r",
        str(GOOGLE_RESEARCH_DIR / "cmmd" / "requirements.txt"),
    ])

if INSTALL_DEPS or INSTALL_TORCH:
    checked_run([
        str(CMMD_PYTHON), "-m", "pip", "install", "--upgrade", TORCH_PACKAGE,
    ])
    print("Installed PyTorch for Scenic CLIP checkpoint conversion.")

if INSTALL_JAX_CUDA:
    checked_run([
        str(CMMD_PYTHON), "-m", "pip", "install", "--upgrade", "--force-reinstall",
        "--no-cache-dir", JAX_CUDA_PACKAGE,
    ])
    print(f"Installed {JAX_CUDA_PACKAGE}. Restart the kernel if it uses this interpreter.")

if INSTALL_DEPS or REPAIR_NUMPY_ABI:
    # TensorFlow/Keras imports optional compiled packages such as Pandas.
    # Reinstall them together so they target the same NumPy ABI.
    checked_run([
        str(CMMD_PYTHON), "-m", "pip", "install", "--upgrade", "--force-reinstall",
        "--no-cache-dir", "--no-deps",
        "pandas==2.2.3",
        "h5py==3.12.1",
        "scipy==1.14.1",
        "scikit-learn==1.5.2",
        "matplotlib==3.9.2",
    ])
    checked_run([
        str(CMMD_PYTHON), "-c",
        (
            "import numpy, pandas, h5py, scipy, sklearn, matplotlib; "
            "print('ABI imports OK:', numpy.__version__, pandas.__version__)"
        ),
    ])
    print("CMMD dependencies repaired. Restart the kernel if it uses this interpreter.")
else:
    print("CMMD requirements install/ABI repair skipped. Set INSTALL_DEPS or REPAIR_NUMPY_ABI to True if needed.")

print(f"CMMD interpreter: {CMMD_PYTHON}")


## 2. Evaluation configuration

Google CMMD officially reads PNG/JPEG images directly inside flat directories. This notebook stages recursive paths as temporary symlinks. The automatically selected batch size divides both image counts, preventing the official loader from dropping a partial final batch.


In [ ]:
REAL_DIR = Path("/path/to/real")
SAMPLES_DIR = Path("/path/to/samples")

MAX_IMAGES = None  # None means all images.
MAX_BATCH_SIZE = 32
CMMD_BATCH_SIZE = None  # None chooses the largest common divisor <= MAX_BATCH_SIZE.
REQUIRE_JAX_GPU = True
ISOLATE_JAX_PIP_CUDA_LIBS = True  # Clears LD_LIBRARY_PATH/CUDA_HOME/CUDA_PATH for JAX pip CUDA wheels.

OUTPUT_PATH = Path("official_cmmd.json")
RUN_CMMD = False


In [ ]:
IMAGE_EXTENSIONS = {".jpeg", ".jpg", ".png"}


def discover_images(root: Path, maximum: int | None) -> list[Path]:
    if not root.is_dir():
        return []
    paths = sorted(
        path.resolve()
        for path in root.rglob("*")
        if path.is_file() and path.suffix.casefold() in IMAGE_EXTENSIONS
    )
    return paths if maximum is None else paths[:maximum]


def common_batch_size(first: int, second: int, limit: int) -> int:
    for candidate in range(min(first, second, limit), 0, -1):
        if first % candidate == 0 and second % candidate == 0:
            return candidate
    return 1


real_paths = discover_images(REAL_DIR.expanduser(), MAX_IMAGES)
sample_paths = discover_images(SAMPLES_DIR.expanduser(), MAX_IMAGES)
cmmd_batch_size = (
    common_batch_size(len(real_paths), len(sample_paths), MAX_BATCH_SIZE)
    if CMMD_BATCH_SIZE is None and real_paths and sample_paths
    else CMMD_BATCH_SIZE
)

def cmmd_subprocess_environment() -> dict[str, str]:
    environment = os.environ.copy()
    environment["PYTHONPATH"] = os.pathsep.join(filter(None, [
        str(GOOGLE_RESEARCH_DIR.resolve()),
        str(SCENIC_DIR.resolve()),
        environment.get("PYTHONPATH", ""),
    ]))
    environment.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
    environment.setdefault("PYTHONNOUSERSITE", "1")
    environment.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
    environment.setdefault("JAX_TRACEBACK_FILTERING", "off")
    if ISOLATE_JAX_PIP_CUDA_LIBS:
        for key in ("LD_LIBRARY_PATH", "CUDA_HOME", "CUDA_PATH"):
            environment.pop(key, None)
    if REQUIRE_JAX_GPU:
        environment["JAX_PLATFORM_NAME"] = "gpu"
    return environment


dependency_environment = cmmd_subprocess_environment()

dependency_script = f"""
import torch
print('torch', torch.__version__)
print('TORCH_AVAILABLE=1')

import os
import numpy, pandas, tensorflow, jax, jaxlib, flax
import jax.numpy as jnp
from cmmd import embedding

print('LD_LIBRARY_PATH_SET=' + ('1' if os.environ.get('LD_LIBRARY_PATH') else '0'))
print('CUDA_HOME_SET=' + ('1' if os.environ.get('CUDA_HOME') else '0'))
print('CUDA_PATH_SET=' + ('1' if os.environ.get('CUDA_PATH') else '0'))

try:
    devices = jax.devices()
except Exception as exc:
    print('JAX_DEVICE_CHECK_ERROR=' + repr(exc))
    print('JAX_GPU_AVAILABLE=0')
    if {REQUIRE_JAX_GPU!r}:
        raise
    devices = []
gpu_devices = [
    device for device in devices
    if getattr(device, 'platform', '').lower() in ('gpu', 'cuda', 'rocm', 'metal')
]
print('numpy', numpy.__version__)
print('pandas', pandas.__version__)
print('tensorflow', tensorflow.__version__)
print('jax', jax.__version__)
print('jaxlib', jaxlib.__version__)
print('jaxlib_file', getattr(jaxlib, '__file__', ''))
print('devices', devices)
print('gpu_devices', gpu_devices)
print('JAX_GPU_AVAILABLE=' + ('1' if gpu_devices else '0'))
if gpu_devices:
    x = jnp.ones((1, 8, 8, 3), dtype=jnp.float32)
    w = jnp.ones((3, 3, 3, 4), dtype=jnp.float32)
    y = jax.lax.conv_general_dilated(
        x, w, (1, 1), 'SAME', dimension_numbers=('NHWC', 'HWIO', 'NHWC')
    )
    y.block_until_ready()
    print('JAX_CUDNN_CONV_SMOKE=1')
else:
    print('JAX_CUDNN_CONV_SMOKE=0')
if {REQUIRE_JAX_GPU!r} and not gpu_devices:
    raise RuntimeError('REQUIRE_JAX_GPU=True but JAX did not expose a GPU device')
"""
dependency_check = subprocess.run(
    [str(CMMD_PYTHON), "-c", dependency_script],
    cwd=GOOGLE_RESEARCH_DIR if GOOGLE_RESEARCH_DIR.is_dir() else None,
    env=dependency_environment,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    check=False,
)

checks = {
    "real_images": len(real_paths),
    "sample_images": len(sample_paths),
    "batch_size": cmmd_batch_size,
    "google_cmmd": (GOOGLE_RESEARCH_DIR / "cmmd" / "main.py").is_file(),
    "scenic": (SCENIC_DIR / "scenic").is_dir(),
    "dependency_imports": dependency_check.returncode == 0,
    "torch_available": "TORCH_AVAILABLE=1" in dependency_check.stdout,
    "cuda_env_isolated": not any(dependency_environment.get(key) for key in ("LD_LIBRARY_PATH", "CUDA_HOME", "CUDA_PATH")),
    "jax_device_check_completed": "JAX_GPU_AVAILABLE=" in dependency_check.stdout,
    "jax_gpu_available": "JAX_GPU_AVAILABLE=1" in dependency_check.stdout,
    "jax_cudnn_conv_smoke": "JAX_CUDNN_CONV_SMOKE=1" in dependency_check.stdout,
    "require_jax_gpu": REQUIRE_JAX_GPU,
    "batch_uses_all_real": bool(cmmd_batch_size and len(real_paths) % cmmd_batch_size == 0),
    "batch_uses_all_samples": bool(cmmd_batch_size and len(sample_paths) % cmmd_batch_size == 0),
}
display(checks)
print(dependency_check.stdout)
if not checks["dependency_imports"] and "CUDNN_STATUS_SUBLIBRARY_VERSION_MISMATCH" in dependency_check.stdout:
    print(
        "JAX can see the GPU, but cuDNN sublibraries are mismatched. Keep "
        "ISOLATE_JAX_PIP_CUDA_LIBS = True, set INSTALL_JAX_CUDA = True, choose "
        "JAX_CUDA_PACKAGE, rerun the install cell, restart the kernel, and rerun preflight."
    )
elif not checks["dependency_imports"]:
    print(
        "CMMD dependency imports failed. If the output contains "
        "'numpy.dtype size changed', set REPAIR_NUMPY_ABI = True and rerun the install cell."
    )
if not checks["torch_available"]:
    print(
        "PyTorch is required by Scenic to convert the downloaded CLIP checkpoint. "
        "Set INSTALL_TORCH = True, rerun the install cell once, then rerun preflight."
    )
if REQUIRE_JAX_GPU and not checks["jax_gpu_available"]:
    if not checks["jax_device_check_completed"]:
        print("JAX GPU check did not complete because dependency imports failed first.")
    else:
        print(
            "JAX is not using GPU. Set INSTALL_JAX_CUDA = True, choose JAX_CUDA_PACKAGE, "
            "rerun the install cell, then restart the kernel and rerun preflight."
        )
if checks["jax_gpu_available"] and not checks["jax_cudnn_conv_smoke"]:
    print(
        "JAX GPU is visible, but the cuDNN convolution smoke test failed. This usually means "
        "system CUDA/cuDNN libraries are mixed with JAX pip wheels. Keep ISOLATE_JAX_PIP_CUDA_LIBS = True "
        "for jax[cuda12]/jax[cuda13], or set it to False only if you intentionally use jax[cuda*-local]."
    )
ready = (
    len(real_paths) >= 2
    and len(sample_paths) >= 2
    and (not REQUIRE_JAX_GPU or checks["jax_gpu_available"])
    and (not REQUIRE_JAX_GPU or checks["jax_cudnn_conv_smoke"])
    and all(checks[key] for key in (
        "google_cmmd", "scenic", "dependency_imports", "batch_uses_all_real", "batch_uses_all_samples"
    ))
)


In [ ]:
cmmd = None


def stage_images(paths: list[Path], directory: Path) -> None:
    directory.mkdir(parents=True, exist_ok=True)
    for index, source in enumerate(paths):
        destination = directory / f"{index:08d}{source.suffix.casefold()}"
        try:
            destination.symlink_to(source)
        except OSError:
            os.link(source, destination)


if not RUN_CMMD:
    print("CMMD skipped. Set RUN_CMMD = True after preflight passes.")
elif not ready:
    raise ValueError("CMMD preflight did not pass.")
else:
    started = time.perf_counter()
    with tempfile.TemporaryDirectory(prefix="official-cmmd-") as temporary_dir:
        temporary_root = Path(temporary_dir)
        staged_real = temporary_root / "real_pooled"
        staged_samples = temporary_root / "samples_pooled"
        stage_images(real_paths, staged_real)
        stage_images(sample_paths, staged_samples)

        environment = cmmd_subprocess_environment()
        command = [
            str(CMMD_PYTHON), "-m", "cmmd.main",
            str(staged_real), str(staged_samples),
            f"--batch_size={cmmd_batch_size}",
            f"--max_count={max(len(real_paths), len(sample_paths))}",
        ]
        completed = subprocess.run(
            command,
            cwd=GOOGLE_RESEARCH_DIR,
            env=environment,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            check=False,
        )
        print(completed.stdout)
        if completed.returncode != 0:
            raise RuntimeError(f"Official CMMD failed with exit code {completed.returncode}")
        match = re.search(r"The CMMD value is:\s*([-+0-9.eE]+)", completed.stdout)
        if match is None:
            raise ValueError("Could not parse CMMD from official output")
        cmmd = float(match.group(1))

    elapsed = time.perf_counter() - started
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    base = OUTPUT_PATH.expanduser().resolve()
    output_path = base.with_name(f"{base.stem}_{timestamp}{base.suffix}")
    payload = {
        "created_at_local": datetime.now().astimezone().isoformat(),
        "metric": "CMMD",
        "value": cmmd,
        "real_dir": str(REAL_DIR.expanduser().resolve()),
        "samples_dir": str(SAMPLES_DIR.expanduser().resolve()),
        "real_images": len(real_paths),
        "sample_images": len(sample_paths),
        "batch_size": cmmd_batch_size,
        "require_jax_gpu": REQUIRE_JAX_GPU,
        "isolate_jax_pip_cuda_libs": ISOLATE_JAX_PIP_CUDA_LIBS,
        "torch_available": checks["torch_available"],
        "jax_gpu_available": checks["jax_gpu_available"],
        "jax_cudnn_conv_smoke": checks["jax_cudnn_conv_smoke"],
        "runtime_seconds": elapsed,
        "official_source": "https://github.com/google-research/google-research/tree/master/cmmd",
        "official_output": completed.stdout,
    }
    output_path.write_text(json.dumps(payload, indent=2) + "\n", encoding="utf-8")
    print(f"Official CMMD: {cmmd:.6f}")
    print(f"Saved: {output_path}")


## Notes

- Only PNG/JPEG files are included because those are the official CMMD loader's supported extensions.
- Scenic needs `torch` for the one-time OpenAI CLIP `.pt` checkpoint conversion. If you see `Could not import torch for CLIP checkpoint conversion`, set `INSTALL_TORCH = True` and rerun the install cell once.
- `Unable to initialize backend 'tpu'` is harmless on a non-TPU machine; the important checks are `jax_gpu_available` and `jax_cudnn_conv_smoke`.
- `REQUIRE_JAX_GPU = True` sets `JAX_PLATFORM_NAME=gpu` for preflight and the official CMMD subprocess, so CPU fallback fails fast.
- `ISOLATE_JAX_PIP_CUDA_LIBS = True` clears `LD_LIBRARY_PATH`, `CUDA_HOME`, and `CUDA_PATH` for CMMD subprocesses. Keep this enabled when using `jax[cuda12]` or `jax[cuda13]` pip wheels to avoid system cuDNN overriding JAX's bundled cuDNN libraries.
- The preflight cell runs a small JAX GPU convolution. If it fails with `CUDNN_STATUS_SUBLIBRARY_VERSION_MISMATCH`, reinstall the selected JAX CUDA wheel with `INSTALL_JAX_CUDA = True`, restart the kernel, and rerun preflight before starting CMMD.
- For NVIDIA systems, set `INSTALL_JAX_CUDA = True` and use `JAX_CUDA_PACKAGE = "jax[cuda13]"` or `"jax[cuda12]"` according to the installed driver/CUDA support, then restart the kernel.
- On multi-device JAX systems, the batch size must also be divisible by the visible JAX device count.
- Compare scores only with identical reference images and official code/model revisions.
